# Experiment 3: Clinical Trial Matching with Jaccard Silver Labels

**Paper**: Engineering Truthiness: A Standard for Pseudo–Ground Truth in Machine Learning Evaluation

**Description**: Large-scale patient–trial matching using Jaccard similarity as silver supervision. Includes threshold calibration, fairness analysis, and SILVER compliance demonstration.

**Requirements**: See `requirements.txt` in the repository root.

**Usage**: Run cells sequentially. All outputs are saved to `outputs/{exp_name}/`.

---

In [ ]:
# =========================
# CONFIGURATION
# =========================
# Public mode: all data is generated synthetically or loaded from public sources.
# To use private data, create config/config_paths_private.py (gitignored).

import os, sys
from pathlib import Path

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Optional: mount Google Drive for private data
    # from google.colab import drive
    # drive.mount('/content/drive')
    pass

# Resolve repo root (works from notebooks/ or repo root)
_nb_dir = Path('.').resolve()
_repo_root = _nb_dir.parent if _nb_dir.name == 'notebooks' else _nb_dir

# Try private config first, fall back to public defaults
try:
    sys.path.insert(0, str(_repo_root / 'config'))
    from config_paths_private import *  # noqa: F403
    CONFIG_MODE = 'private'
    print('[INFO] Using private path configuration')
except ImportError:
    # Public defaults — everything runs with synthetic data
    REPO_ROOT = _repo_root
    RUN_ID = 'public_run'
    DATADIR = REPO_ROOT / 'data'
    OUTDIR = REPO_ROOT / 'outputs'
    CONFIG_MODE = 'public'
    print('[INFO] Using public path configuration')

Path(OUTDIR).mkdir(parents=True, exist_ok=True)


In [ ]:
# =========================
# SHARED UTILITIES
# =========================
import os, json, time, platform, hashlib, random
from pathlib import Path
import numpy as np
import pandas as pd

def set_global_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

def now_utc_compact():
    import datetime
    return datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S_utc")

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def bootstrap_ci(x, n=1000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return (np.nan, np.nan, np.nan)
    stats = []
    for _ in range(n):
        samp = rng.choice(x, size=len(x), replace=True)
        stats.append(np.mean(samp))
    lo = np.quantile(stats, alpha/2)
    hi = np.quantile(stats, 1 - alpha/2)
    return (float(np.mean(x)), float(lo), float(hi))

def make_run_dirs(exp_name: str):
    run_id = f"{exp_name}_{now_utc_compact()}"
    base = Path("outputs") / exp_name / run_id
    figs = base / "figures"
    tabs = base / "tables"
    logs = base / "logs"
    for d in [base, figs, tabs, logs]:
        safe_mkdir(d)
    return run_id, base, figs, tabs, logs

def write_manifest(base: Path, meta: dict):
    mf = base / "run_manifest.json"
    meta = dict(meta)
    meta["platform"] = {
        "python": platform.python_version(),
        "platform": platform.platform()
    }
    mf.write_text(json.dumps(meta, indent=2))
    return mf

set_global_seed(42)


In [ ]:
# =========================
# 03 — DATA LOADING (PRIVATE VIA config_paths.py, ELSE PUBLIC SYNTHETIC)
# =========================
EXP = "03_clinical"
run_id, out_base, out_figs, out_tabs, out_logs = make_run_dirs(EXP)

USE_PRIVATE = False
try:
    import config_paths_private as CP
    # Verify private config has the required clinical data paths
    _ = CP.PPATF, CP.PCTTI, CP.PGOLD
    USE_PRIVATE = True
    print("[INFO] Private clinical data paths found")
except (ImportError, AttributeError):
    USE_PRIVATE = False
    print("[INFO] Using synthetic clinical data (public mode)")

USE_PRIVATE


In [ ]:
# =========================
# 03 — LOAD DATAFRAMES
# Expected columns (private mode):
#   - gold_label (0/1)   [if you have anchors; if not, we treat as silver-only and skip calibration]
#   - jaccard (float)
#   - subgroup columns (e.g., gender, ethnicity) OR booleans like is_female, is_male
# =========================
if USE_PRIVATE:
    # These are your private parquets from Drive
    df_pat = pd.read_parquet(CP.PPATF)
    df_trials = pd.read_parquet(CP.PCTTI)
    df_pairs = pd.read_parquet(CP.PGOLD)

    # CRITICAL: FIX THE CLASSIC BUG (merge ordering / missing columns)
    # Ensure df_pairs has patient attributes for subgroup analysis
    key = "patient_id" if "patient_id" in df_pairs.columns else ("unitstayid" if "unitstayid" in df_pairs.columns else None)
    if key is None:
        raise ValueError("Could not find patient key column in df_pairs (expected patient_id or unitstayid).")

    if key in df_pat.columns and key not in df_pairs.columns:
        raise ValueError(f"Key {key} not found in df_pairs, cannot merge.")

    if key in df_pat.columns:
        # only bring subgroup columns (avoid huge joins)
        subgroup_cols = [c for c in df_pat.columns if c.lower() in ["gender", "sex", "ethnicity", "race"] or c.startswith("is_")]
        df_pairs = df_pairs.merge(df_pat[[key] + subgroup_cols], on=key, how="left")

else:
    # PUBLIC FALLBACK: synthetic clinical pairs with subgroup + jaccard
    rng = np.random.default_rng(42)
    n = 200000
    df_pairs = pd.DataFrame({
        "patient_id": rng.integers(0, 20000, size=n),
        "jaccard": np.clip(rng.normal(loc=0.02, scale=0.01, size=n), 0, 0.2),
        "gender": rng.choice(["female","male"], size=n, p=[0.52,0.48]),
        "ethnicity": rng.choice(["majority","minority"], size=n, p=[0.8,0.2]),
    })
    # create a "pseudo-gold" anchor for calibration demo
    # gold=1 if jaccard is high and minority is slightly noisier
    logits = 12*(df_pairs["jaccard"].values - 0.02) + (df_pairs["ethnicity"].values=="minority")*(-0.4)
    p = 1/(1+np.exp(-logits))
    df_pairs["gold_label"] = (rng.random(n) < p).astype(int)

df_pairs.head(), df_pairs.shape


In [ ]:
# =========================
# 03 — THRESHOLD SWEEP (OPERATIONAL CONSTRAINTS)
# Key deliverable: prove τ=0.02 is NOT arbitrary:
# - controls positive fraction near target (~1/3 of candidate set)
# - avoids collapse at higher τ
# =========================
taus = np.round(np.arange(0.01, 0.061, 0.002), 3)  # focus around your operating region

target_pos_frac = 1/3  # your stated objective
min_pos_frac = 0.05    # non-degenerate
max_pos_frac = 0.60    # too noisy

rows = []
for tau in taus:
    silver = (df_pairs["jaccard"].values >= tau).astype(int)
    pos_frac = silver.mean()
    rows.append({"tau": tau, "pos_frac": pos_frac})

df_tau = pd.DataFrame(rows)
df_tau["dist_to_target"] = (df_tau["pos_frac"] - target_pos_frac).abs()
df_tau.sort_values("dist_to_target").head(10)


In [ ]:
# =========================
# 03 — CALIBRATION + FAIRNESS (IF gold_label EXISTS)
# =========================
has_gold = "gold_label" in df_pairs.columns
has_gender = "gender" in df_pairs.columns

fair_rows = []
if has_gold:
    for tau in taus:
        df = df_pairs.copy()
        df["silver"] = (df["jaccard"] >= tau).astype(int)

        # alpha,beta style rates (proxy; your paper can define precisely)
        # alpha = P(silver=1|gold=0) = FP rate under gold
        # beta  = P(silver=0|gold=1) = FN rate under gold
        gold0 = df["gold_label"] == 0
        gold1 = df["gold_label"] == 1
        alpha = df.loc[gold0, "silver"].mean() if gold0.any() else np.nan
        beta  = (1 - df.loc[gold1, "silver"].mean()) if gold1.any() else np.nan

        # subgroup FN gap (gender as example)
        fn_gap = np.nan
        if has_gender:
            for g in ["female","male"]:
                pass
            sub = {}
            for g in df["gender"].dropna().unique():
                dfg = df[df["gender"] == g]
                gold1g = dfg["gold_label"] == 1
                if gold1g.any():
                    fnr = (1 - dfg.loc[gold1g, "silver"].mean())
                    sub[g] = fnr
            if len(sub) >= 2:
                fn_gap = max(sub.values()) - min(sub.values())

        fair_rows.append({
            "tau": tau,
            "pos_frac": df["silver"].mean(),
            "alpha_fp_rate": alpha,
            "beta_fn_rate": beta,
            "fn_gap_gender": fn_gap
        })

df_tau_fair = pd.DataFrame(fair_rows) if fair_rows else None
df_tau_fair.head() if df_tau_fair is not None else "No gold_label; skipping calibration block."


In [ ]:
# =========================
# 03 — CHOOSE OPERATING τ + EXPORT TABLE
# =========================
tau_star = 0.02
if tau_star not in set(df_tau["tau"].values):
    tau_star = float(df_tau.loc[df_tau["dist_to_target"].idxmin(), "tau"])

summary = df_tau[df_tau["tau"].isin([0.015, 0.018, tau_star, 0.025, 0.03])].copy()
summary = summary.sort_values("tau")

if df_tau_fair is not None:
    summary = summary.merge(df_tau_fair, on=["tau","pos_frac"], how="left")

summary.to_csv(out_tabs / "results_03_threshold_sweep_summary.csv", index=False)
summary


In [ ]:
# =========================
# 03 — PLOT THRESHOLD TRADEOFF
# =========================
import matplotlib.pyplot as plt

plt.figure(figsize=(7,4))
plt.plot(df_tau["tau"], df_tau["pos_frac"], marker="o")
plt.axhline(target_pos_frac, linestyle="--")
plt.axvline(tau_star, linestyle="--")
plt.xlabel("Jaccard threshold (τ)")
plt.ylabel("Positive fraction (silver=1)")
plt.title(f"Operational justification of τ (target ≈ {target_pos_frac:.2f}, chosen τ={tau_star:.3f})")
plt.grid(True)
plt.tight_layout()
plt.savefig(out_figs / "fig01_tau_vs_posfrac.png", dpi=300)

manifest = write_manifest(out_base, {
    "experiment": EXP,
    "run_id": run_id,
    "use_private": USE_PRIVATE,
    "tau_star": tau_star,
    "target_pos_frac": target_pos_frac,
    "has_gold": bool(has_gold),
    "outputs": {
        "tables": [str(out_tabs / "results_03_threshold_sweep_summary.csv")],
        "figures": [str(out_figs / "fig01_tau_vs_posfrac.png")]
    }
})
manifest


In [ ]:
from IPython.display import Image, display as ipy_display
import glob

print("Saved figures:")
for p in sorted(glob.glob(str(out_figs / "*.png"))):
    print(" -", p)
    ipy_display(Image(filename=p))


In [ ]:
# ==============================
# END OF NOTEBOOK SUMMARY
# ==============================
import json
summary = {
    "run_dir": str(out_base),
    "fig_dir": str(out_figs),
    "tables_dir": str(out_tabs),
}
summary_path = out_base / "run_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print("[DONE] Outputs saved under:", out_base)
